# 09 — Seleksi, lima seed final, dan laporan

Seleksi hanya setelah semua **270 kasus CV** lengkap. Pilih arm+LR+batch per model menurut mean macro-F1, lalu SD, nama arm, ID skenario. Model utama juga dipilih dari CV. CV hasil seleksi bersifat optimistis, bukan nested CV.

Epoch final = median best epoch lima fold. Lima seed [42,43,44,45,46] ditetapkan sebelum test; seed 42 primer. Lima seed menghasilkan lima model per arsitektur untuk mengukur variasi, bukan memilih seed terbaik atau otomatis membuat ensemble. Tidak ada k-fold pada test.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "research_v2").is_dir() else Path.cwd().parent
assert (ROOT / "research_v2").is_dir(), "Buka notebook dari folder project/notebooks"
sys.path.insert(0, str(ROOT / "notebooks"))
from _research_runner import ResearchRunner
RUN_ID = "20260930_anatomy_fs2000_v2"
DEVICE = "mps"  # Mac Apple Silicon; ubah ke "cpu" bila perlu.
runner = ResearchRunner(ROOT, RUN_ID, DEVICE, label="v2_manual")
RUN = ROOT / "results/research_v2" / RUN_ID


In [ ]:
runner.run_stage("select")
display(pd.read_csv(RUN / "reports/cv_summary.csv"))

In [ ]:
table = pd.read_csv(RUN / "reports/cv_summary.csv")
for model, part in table.groupby("model"):
    display(part.pivot(index="scenario", columns="arm", values="f1_mean").style.format("{:.3f}").background_gradient(cmap="Blues"))

In [ ]:
runner.run_stage("final", "--model", "encoder")

In [ ]:
runner.run_stage("final", "--model", "patchtst")

In [ ]:
runner.run_stage("final", "--model", "hierarchical")

## Evaluasi hold-out historis

Tahap berikut terkunci hingga 15 final selesai. Baru saat ini 59 pasien test dipreproses dengan resep yang sama, scaler development, lalu diinferensi. Dataset pernah dievaluasi pada pekerjaan lama: tidak disebut test baru/validasi eksternal. Jangan memilih model, seed, atau metode berdasarkan angka ini.

In [ ]:
runner.run_stage("test")
runner.run_stage("report")

In [ ]:
display(pd.read_csv(RUN / "reports/historical_test_all_seeds.csv"))
display(pd.read_csv(RUN / "reports/seed_variation.csv"))
display(Image(filename=str(RUN / "reports/historical_test_uncertainty.png")))

CI 95% memakai 2.000 bootstrap **pasien**, terstratifikasi kelas, pada setiap seed secara terpisah. SD antar-seed menggambarkan variasi training; jangan menganggap 59×5 sebagai pasien independen. CI tidak mencakup seluruh ketidakpastian pemilihan desain.

Tahap penelitian sesudah ini: GAN delapan kanal independen dan pemeriksaan morfologi/keragaman pada training saja; perbandingan augmentasi terkontrol pada preprocessing V2; segmentasi PVC teranotasi; validasi pasien baru. Jangan menambahkan arm setelah melihat test ke dalam run yang sudah dibekukan.